<a href="https://colab.research.google.com/github/aditi19sahu/eRes/blob/main/India_Oil_Inflation_PassThrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oil → Inflation Pass-Through in India
Monthly regressions of Indian price indices on rupee crude-oil price changes, Jan 2013 – Sep 2026. Work in progress.

Pass-through is traced along the price chain: crude → wholesale fuel (WPI) → retail fuel items (CPI) → CPI groups → core → headline.
Data: MoSPI CPI and WPI APIs, FRED (IMF/EIA crude, INR/USD), PPAC Indian crude basket, RBI Handbook Table 40 (repo rate). Raw files are in `data/`.

## 1. Settings

In [ ]:
#@title Settings { display-mode: "form" }
DATA_DIR = "data"  #@param {type:"string"}
START = "2013-01"  #@param {type:"string"}
END = ""  #@param {type:"string"}
OIL_LAGS = 6  #@param {type:"slider", min:0, max:12, step:1}
AR_LAGS = 2  #@param {type:"slider", min:0, max:6, step:1}
HAC_LAGS = 6  #@param {type:"slider", min:0, max:12, step:1}
ROLL_WINDOW = 60  #@param {type:"slider", min:36, max:120, step:12}

## 2. Estimator
Sum of oil-lag coefficients = cumulative pass-through over OIL_LAGS months; HAC standard errors. Validated on simulated data.

In [ ]:
import os, numpy as np, pandas as pd, statsmodels.api as sm
import plotly.graph_objects as go, plotly.io as pio
pio.renderers.default = "colab"

def fit(d, y, x="oil", L=None, AR=None, H=None, controls=("d_repo", "splice_2025m1", "splice_2026m1")):
    L, AR, H = (OIL_LAGS if L is None else L), (AR_LAGS if AR is None else AR), (HAC_LAGS if H is None else H)
    X = pd.DataFrame(index=d.index)
    for k in range(L + 1):
        X[f"oil_l{k}"] = d[x].shift(k)
    for k in range(1, AR + 1):
        X[f"y_l{k}"] = d[y].shift(k)
    for c in controls:
        if d[c].loc[d[y].notna()].std() > 0:
            X[c] = d[c]
    X = pd.concat([X, pd.get_dummies(d.index.month, prefix="m", drop_first=True, dtype=float).set_index(d.index)], axis=1)
    Z = pd.concat([d[y], X], axis=1).dropna()
    res = sm.OLS(Z[y], sm.add_constant(Z.drop(columns=y))).fit(cov_type="HAC", cov_kwds={"maxlags": H})
    w = np.array([1.0 if n.startswith("oil_l") else 0.0 for n in res.params.index])
    b, se = float(w @ res.params.values), float(np.sqrt(w @ res.cov_params().values @ w))
    rho = float(res.params[[f"y_l{k}" for k in range(1, AR + 1)]].sum()) if AR else 0.0
    return dict(res=res, b=b, se=se, lo=b - 1.96 * se, hi=b + 1.96 * se, long_run=b / (1 - rho), n=int(res.nobs),
                window=f"{Z.index.min():%Y-%m} to {Z.index.max():%Y-%m}")

rng = np.random.default_rng(0)
sim = pd.DataFrame(index=pd.period_range("1980-01", periods=600, freq="M").to_timestamp())
sim["oil"] = rng.normal(0, 8, len(sim))
true_b = np.array([0.006, 0.005, 0.004, 0.003, 0.001, 0.001, 0.0])
sim["y"] = 0.4 + rng.normal(0, 0.1, len(sim)) + sum(b * sim["oil"].shift(k).fillna(0) for k, b in enumerate(true_b))
sim["d_repo"] = sim["splice_2025m1"] = sim["splice_2026m1"] = 0.0
t = fit(sim, "y", L=6, AR=0, H=6)
assert abs(t["b"] - true_b.sum()) < 3 * t["se"]
print(f"Simulated: true {true_b.sum():.4f}, estimated {t['b']:.4f} (95% CI {t['lo']:.4f} to {t['hi']:.4f})")

Simulated: true 0.0200, estimated 0.0218 (95% CI 0.0190 to 0.0246)


## 3. Load data

In [ ]:
def rd(f):
    d = pd.read_csv(os.path.join(DATA_DIR, f), index_col=0)
    d.index = pd.to_datetime(d.index).to_period("M").to_timestamp()
    return d

cpi = rd("cpi_spliced_subindices.csv")
cpi_old = rd("cpi_base2012_groups.csv")
cpi_new = rd("cpi_base2024_divisions.csv")
items = rd("cpi_base2012_items.csv")
wpi = rd("wpi_spliced.csv")
wpi_new = rd("wpi_base2022_23.csv")
crude = rd("crude_prices.csv")
icb_check = pd.read_csv(os.path.join(DATA_DIR, "icb_proxy_validation.csv"))
r = pd.read_csv(os.path.join(DATA_DIR, "repo_rate.csv"), parse_dates=["date"]).sort_values("date")
repo = r.set_index("date")["rate"].resample("D").ffill().resample("MS").last()
for name, d in [("CPI spliced", cpi), ("CPI items", items), ("WPI spliced", wpi), ("Crude", crude.dropna(subset=["icb_chained"]))]:
    print(f"{name:12s} {d.index.min():%Y-%m} to {d.index.max():%Y-%m}")

CPI spliced  2013-01 to 2026-09
CPI items    2014-01 to 2025-12
WPI spliced  2013-01 to 2026-09
Crude        2013-01 to 2026-10


## 4. Data checks
Published figures the data must reproduce.

In [ ]:
yy = lambda s, m: 100 * (s[m] / s[m - pd.DateOffset(years=1)] - 1)
checks = [
    ("CPI headline y/y, Sep 2026", yy(cpi["headline"], pd.Timestamp("2026-09-01")), 5.54, "MoSPI release"),
    ("CPI headline y/y, Dec 2024", yy(cpi_old["general"], pd.Timestamp("2024-12-01")), 5.22, "MoSPI release (base 2012)"),
    ("Core CPI y/y, Aug 2026 (base 2024)", yy(cpi_new["core"], pd.Timestamp("2026-08-01")), 4.2, "RBI MPC statement, 7 Oct 2026"),
    ("WPI y/y, Sep 2026 (base 2022-23)", yy(wpi_new["wpi"], pd.Timestamp("2026-09-01")), 11.39, "OEA release"),
    ("WPI fuel & power y/y, Sep 2026", yy(wpi_new["fuel_power"], pd.Timestamp("2026-09-01")), 25.85, "OEA release"),
    ("Repo rate, Oct 2026", repo["2026-10-01"], 5.50, "RBI MPC statement"),
]
chk = pd.DataFrame(checks, columns=["check", "computed", "published", "source"])
chk["ok"] = (chk.computed - chk.published).abs() < 0.06
assert chk.ok.all(), chk
display(chk.round(2))
print("Indian crude basket proxy (Dated Brent) vs PPAC official:")
display(icb_check.round(2))

,check,computed,published,source,ok
0,"CPI headline y/y, Sep 2026",5.54,5.54,MoSPI release,True
1,"CPI headline y/y, Dec 2024",5.22,5.22,MoSPI release (base 2012),True
2,"Core CPI y/y, Aug 2026 (base 2024)",4.19,4.20,"RBI MPC statement, 7 Oct 2026",True
3,"WPI y/y, Sep 2026 (base 2022-23)",11.39,11.39,OEA release,True
4,"WPI fuel & power y/y, Sep 2026",25.85,25.85,OEA release,True
5,"Repo rate, Oct 2026",5.50,5.50,RBI MPC statement,True


Indian crude basket proxy (Dated Brent) vs PPAC official:


,period,official,proxy,err_pct
0,FY14-15,84.16,85.46,1.55
1,FY15-16,46.17,47.30,2.44
2,FY16-17,47.56,48.54,2.07
3,FY17-18,56.43,57.53,1.95
4,FY18-19,69.88,70.15,0.39
5,FY19-20,60.47,61.18,1.17
6,FY20-21,44.82,44.35,-1.04
7,FY21-22,79.18,80.55,1.73
8,2026-01,63.08,66.60,5.58
9,2026-02,69.01,70.89,2.72


## 5. Build monthly dataset
All variables are 100 × monthly log changes. Oil = Indian crude basket (proxy chained to PPAC official from Feb 2026) × INR/USD.

In [ ]:
dl = lambda s: 100 * np.log(s).diff()
df = pd.DataFrame(index=cpi.index)
df["oil"] = dl(crude["icb_chained"] * crude["inr_per_usd"])
df["oil_usd"] = dl(crude["icb_chained"])
df["d_fx"] = dl(crude["inr_per_usd"])
for c in cpi:
    df[f"cpi_{c}"] = dl(cpi[c])
for c in ["petrol", "diesel", "lpg"]:
    df[f"item_{c}"] = dl(items[c])
for c in wpi:
    df[f"wpi_{c}"] = dl(wpi[c])
df["d_repo"] = repo.diff().shift(1).reindex(df.index)
df["splice_2025m1"] = (df.index == "2025-01-01").astype(float)
df["splice_2026m1"] = (df.index == "2026-01-01").astype(float)
df = df.loc[START:(END or None)]
df.describe().T[["count", "mean", "std"]].round(2)

,count,mean,std
oil,164.0,0.39,11.63
oil_usd,164.0,0.05,11.73
d_fx,165.0,0.34,1.30
cpi_headline,164.0,0.42,0.66
cpi_food,164.0,0.44,1.26
cpi_fuel,164.0,0.35,0.84
cpi_transport,164.0,0.35,0.72
cpi_core,164.0,0.41,0.25
item_petrol,140.0,0.19,2.35
item_diesel,140.0,0.32,2.70


## 6. Pass-through along the price chain
Same specification for every outcome: oil lags 0–6, AR(2), month dummies, lagged Δrepo, splice dummies.

In [ ]:
LADDER = [("WPI mineral oils", "wpi_mineral_oils"), ("WPI fuel & power", "wpi_fuel_power"), ("WPI all commodities", "wpi_wpi"),
          ("CPI item: petrol", "item_petrol"), ("CPI item: diesel", "item_diesel"), ("CPI item: LPG", "item_lpg"),
          ("CPI fuel & light", "cpi_fuel"), ("CPI transport", "cpi_transport"), ("CPI food", "cpi_food"),
          ("CPI core (ex food & fuel)", "cpi_core"), ("CPI headline", "cpi_headline")]
est = {name: fit(df, y) for name, y in LADDER}
ladder = pd.DataFrame({k: {c: v[c] for c in ["b", "se", "lo", "hi", "long_run", "n", "window"]} for k, v in est.items()}).T
display(ladder.round(4))

fig = go.Figure()
names = list(ladder.index)[::-1]
fig.add_trace(go.Scatter(x=ladder.loc[names, "b"], y=names, mode="markers", marker=dict(size=9, color="#2a78d6"),
                         error_x=dict(type="data", symmetric=False, array=(ladder.loc[names, "hi"] - ladder.loc[names, "b"]).astype(float),
                                      arrayminus=(ladder.loc[names, "b"] - ladder.loc[names, "lo"]).astype(float), color="#2a78d6"), name="Pass-through"))
fig.add_vline(x=0, line=dict(color="#898781", width=1))
fig.update_layout(title=f"Cumulative pass-through of a 1% rupee crude move over {OIL_LAGS + 1} months (95% CI)",
                  xaxis_title="% change in index per 1% change in rupee crude", height=480, paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
                  margin=dict(l=200))
fig.show()

,b,se,lo,hi,long_run,n,window
WPI mineral oils,0.747722,0.101082,0.549601,0.945844,0.648781,158,2013-08 to 2026-09
WPI fuel & power,0.40819,0.060567,0.289478,0.526902,0.39818,158,2013-08 to 2026-09
WPI all commodities,0.086217,0.014884,0.057044,0.11539,0.10014,158,2013-08 to 2026-09
CPI item: petrol,0.220066,0.043951,0.133922,0.306209,0.197781,136,2014-04 to 2025-12
CPI item: diesel,0.244582,0.051409,0.143821,0.345343,0.227352,136,2014-04 to 2025-12
CPI item: LPG,0.150595,0.057688,0.037527,0.263664,0.211903,136,2014-04 to 2025-12
CPI fuel & light,0.050124,0.016306,0.018165,0.082083,0.060879,158,2013-08 to 2026-09
CPI transport,0.030479,0.01951,-0.007762,0.068719,0.034484,158,2013-08 to 2026-09
CPI food,-0.017032,0.014156,-0.044777,0.010713,-0.018089,158,2013-08 to 2026-09
CPI core (ex food & fuel),0.002807,0.004273,-0.005569,0.011182,0.003699,158,2013-08 to 2026-09


## 7. Headline: food noise and direct vs indirect effects
Food control removes weather-driven noise. Controlling for core isolates the direct effect (fuel items) and removes the indirect channel (costs passed into core prices), so it is a decomposition, not a better estimate.

In [ ]:
df["food_ctrl"] = df["cpi_food"]
df["core_ctrl"] = df["cpi_core"]
base = ("d_repo", "splice_2025m1", "splice_2026m1")
specs = {
    "Headline": fit(df, "cpi_headline"),
    "Headline, food-inflation control": fit(df, "cpi_headline", controls=base + ("food_ctrl",)),
    "Headline, core-inflation control (direct effect only)": fit(df, "cpi_headline", controls=base + ("core_ctrl",)),
    "Headline, oil in USD + FX separately": fit(df, "cpi_headline", x="oil_usd", controls=base + ("d_fx",)),
}
tab = pd.DataFrame({k: {c: v[c] for c in ["b", "se", "lo", "hi", "n"]} for k, v in specs.items()}).T
display(tab.round(4))
W_PETROL, W_DIESEL = 2.19, 0.15
mech = (W_PETROL * est["CPI item: petrol"]["b"] + W_DIESEL * est["CPI item: diesel"]["b"]) / 100
print(f"Mechanical direct effect = CPI weights (petrol {W_PETROL}%, diesel {W_DIESEL}%) × item pass-through = {mech:.4f}")
print(f"→ a 10% rise in rupee crude adds ≈ {1000 * mech:.0f} bps to headline CPI through pump prices alone.")

,b,se,lo,hi,n
Headline,-0.0036,0.0069,-0.0173,0.0100,158.0
"Headline, food-inflation control",0.0059,0.0033,-0.0005,0.0123,158.0
"Headline, core-inflation control (direct effect only)",-0.0082,0.0071,-0.0220,0.0057,158.0
"Headline, oil in USD + FX separately",-0.0039,0.0067,-0.0171,0.0092,158.0


Mechanical direct effect = CPI weights (petrol 2.19%, diesel 0.15%) × item pass-through = 0.0052
→ a 10% rise in rupee crude adds ≈ 5 bps to headline CPI through pump prices alone.


## 8. Pricing regimes
Petrol/diesel pricing: fortnightly revisions (to May 2017), daily dynamic pricing (Jun 2017 – Mar 2022), effectively frozen (Apr 2022 onward, revised in May 2026).

In [ ]:
REGIMES = [("Fortnightly pricing", "2013-01", "2017-05"), ("Daily pricing", "2017-06", "2022-03"), ("Frozen / administered", "2022-04", "2026-09")]
rows = []
for y_name, y in [("WPI mineral oils", "wpi_mineral_oils"), ("CPI item: petrol", "item_petrol"), ("CPI item: diesel", "item_diesel"), ("CPI headline", "cpi_headline")]:
    for reg, a, b in REGIMES:
        e = fit(df.loc[a:b], y, AR=1, H=4)
        rows.append(dict(outcome=y_name, regime=reg, b=e["b"], lo=e["lo"], hi=e["hi"], n=e["n"], window=e["window"]))
regimes = pd.DataFrame(rows).set_index(["outcome", "regime"])
display(regimes.round(4))

b      lo      hi   n  \
outcome          regime                                              
WPI mineral oils Fortnightly pricing    0.7969  0.6341  0.9597  46   
                 Daily pricing          0.5944  0.4542  0.7346  52   
                 Frozen / administered  0.7180  0.4781  0.9579  48   
CPI item: petrol Fortnightly pricing    0.1952  0.0535  0.3369  39   
                 Daily pricing          0.2686  0.0803  0.4568  48   
                 Frozen / administered -0.0168 -0.0426  0.0090  39   
CPI item: diesel Fortnightly pricing    0.1714  0.0763  0.2666  39   
                 Daily pricing          0.4033  0.2059  0.6006  48   
                 Frozen / administered -0.0183 -0.0476  0.0110  39   
CPI headline     Fortnightly pricing   -0.0236 -0.0506  0.0035  46   
                 Daily pricing          0.0102 -0.0136  0.0341  52   
                 Frozen / administered  0.0134 -0.0066  0.0335  48   

                                                    window  
outcome          regime                                     
WPI mineral oils Fortnightly pricing    2013-08 to 2017-05  
                 Daily pricing          2017-12 to 2022-03  
                 Frozen / administered  2022-10 to 2026-09  
CPI item: petrol Fortnightly pricing    2014-03 to 2017-05  
                 Daily pricing          2017-12 to 2022-03  
                 Frozen / administered  2022-10 to 2025-12  
CPI item: diesel Fortnightly pricing    2014-03 to 2017-05  
                 Daily pricing          2017-12 to 2022-03  
                 Frozen / administered  2022-10 to 2025-12  
CPI headline     Fortnightly pricing    2013-08 to 2017-05  
                 Daily pricing          2017-12 to 2022-03  
                 Frozen / administered  2022-10 to 2026-09

## 9. Rolling-window headline pass-through (food-controlled)

In [ ]:
months = df.dropna(subset=["cpi_headline", "oil"]).index
rows = []
for i in range(ROLL_WINDOW, len(months) + 1):
    w = df.loc[months[i - ROLL_WINDOW]:months[i - 1]]
    e = fit(w, "cpi_headline", controls=base + ("food_ctrl",))
    rows.append(dict(end=months[i - 1], b=e["b"], lo=e["lo"], hi=e["hi"]))
roll = pd.DataFrame(rows).set_index("end")
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(roll.index) + list(roll.index[::-1]), y=list(roll.hi) + list(roll.lo[::-1]), fill="toself",
                         fillcolor="rgba(42,120,214,0.15)", line=dict(width=0), name="95% CI", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=roll.index, y=roll.b, mode="lines", line=dict(color="#2a78d6", width=2), name="Pass-through"))
fig.add_hline(y=0, line=dict(color="#898781", width=1))
fig.update_layout(title=f"Rolling {ROLL_WINDOW}-month headline pass-through, food-controlled", xaxis_title="Window end",
                  yaxis_title="pp CPI per 1% rupee crude", height=420, paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb")
fig.show()
display(roll.iloc[::12].round(4))

,b,lo,hi
end,,,
2018-01-01,0.0019,-0.0029,0.0067
2019-01-01,0.0029,-0.0043,0.0102
2020-01-01,0.0091,-0.0004,0.0187
2021-01-01,0.0017,-0.0075,0.0109
2022-01-01,0.0107,0.0002,0.0211
2023-01-01,0.0100,0.0023,0.0177
2024-01-01,0.0087,0.0026,0.0148
2025-01-01,0.0207,0.0108,0.0306
2026-01-01,0.0150,0.0065,0.0234


## 10. Diagnostics

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = pd.DataFrame({k: acorr_ljungbox(v["res"].resid, lags=[12], return_df=True)["lb_pvalue"].iloc[0] for k, v in est.items()}, index=["Ljung-Box p (12 lags)"]).T
display(lb.round(3))

,Ljung-Box p (12 lags)
WPI mineral oils,0.255
WPI fuel & power,0.371
WPI all commodities,0.556
CPI item: petrol,0.737
CPI item: diesel,0.449
CPI item: LPG,0.524
CPI fuel & light,0.836
CPI transport,0.750
CPI food,0.802
CPI core (ex food & fuel),0.860


## 11. Next round
EV interaction (Δoil × EV share, VAHAN); official PPAC Indian-basket history (registration-gated download); base-2024 item indices for petrol/diesel in 2026; excise-duty series as an explicit policy-wedge variable.